In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

caminho = '/content/drive/My Drive/DATA COMEXSTAT/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Função auxiliar para carregar múltiplos CSVs rapidamente
def carregar_arquivos(nomes, prefixo=""):
    return {
        nome.lower(): pd.read_csv(f"{caminho}{prefixo}{nome.upper()}.csv", sep=";", encoding="latin1")
        for nome in nomes
    }


In [ ]:
# Arquivos para tratamento dados do estado
arquivos_estado = ["Exp_2025", "Exp_2026", "Imp_2025", "Imp_2026"]
arquivos_auxiliares = ["ncm", "pais", "via", "urf"]

# Carregar os dados de estado
dados_estado = carregar_arquivos(arquivos_estado)

# Carregar dados auxiliares
dados_auxiliares = carregar_arquivos(arquivos_auxiliares)

# Verificar as primeiras linhas dos dados carregados
print(dados_estado["exp_2026"].head())


   CO_ANO  CO_MES    CO_NCM  CO_UNID  CO_PAIS SG_UF_NCM  CO_VIA  CO_URF  \
0    2026       1  96151100       10      325        AP       7  240151   
1    2026       1   8045020       10      697        PR       1  917800   
2    2026       1  87081000       11      521        SP       1  817800   
3    2026       1   7133210       10      232        MA       1  317903   
4    2026       1   3075200       10      467        SP       1  817800   

   QT_ESTAT  KG_LIQUIDO  VL_FOB  
0         9           9      67  
1        55          55      47  
2        11          25     583  
3         5           5      24  
4        10          10     144  


In [ ]:
# Função para processar os dados de estado
def processar_estado(dados_estado, dados_auxiliares):
    df_estado = pd.concat([dados_estado[f"exp_{ano}"] for ano in range(2025, 2026)], ignore_index=True)
    df_estado = pd.concat([df_estado, pd.concat([dados_estado[f"imp_{ano}"] for ano in range(2025, 2026)], ignore_index=True)], ignore_index=True)

    # Renomear colunas conforme o dicionário
    renomear_colunas_estado = {
        "CO_ANO":"Ano",
        "CO_MES":"Mês",
        "CO_NCM":"Produto",
        "CO_UNID":"Unidade",
        "CO_PAIS":"País",
        "SG_UF_NCM":"Estado",
        "CO_VIA":"Modal",
        "CO_URF":"Aduana",
        "QT_ESTAT":"Quantidade",
        "KG_LIQUIDO":"Peso",
        "VL_FOB":"Valor US$",
    }
    df_estado = df_estado.rename(columns=renomear_colunas_estado)


    pais_lookup = dados_auxiliares["pais"][["CO_PAIS", "NO_PAIS"]].rename(columns={"CO_PAIS": "País", "NO_PAIS": "Nome do País"})
    df_estado = df_estado.merge(pais_lookup, on="País", how="left")
    ncm_lookup = dados_auxiliares["ncm"][["CO_NCM", "NO_NCM_POR"]].rename(columns={"CO_NCM": "Produto", "NO_NCM_POR": "Nome do Produto"})
    df_estado = df_estado.merge(ncm_lookup, on="Produto", how="left")
    via_lookup = dados_auxiliares["via"][["CO_VIA", "NO_VIA"]].rename(columns={"CO_VIA": "Modal", "NO_VIA": "Nome do Modal"})
    df_estado = df_estado.merge(via_lookup, on="Modal", how="left")
    urf_lookup = dados_auxiliares['urf'][["CO_URF", "NO_URF"]].rename(columns={"CO_URF": "Aduana", "NO_URF": "Nome da Aduana"})
    df_estado = df_estado.merge(urf_lookup, on="Aduana", how="left")

    return df_estado

# Processar estado
df_estado_processado = processar_estado(dados_estado, dados_auxiliares)

# Salvar arquivo processado
df_estado_processado.to_csv('/content/drive/My Drive/DATA COMEXSTAT/Análise_2025-2026.csv', index=False)

In [ ]:
# Visualização do data frame
print(df_estado_processado.head())

    Ano  Mês   Produto  Unidade  País Estado  Modal   Aduana  Quantidade  \
0  2025    8  71039900       19   687     RS      4   817700       61690   
1  2025   12   2071412       10   741     SC      1   817800       83835   
2  2025   11  33059000       10   767     SP      4   817600         792   
3  2025   11  56031330       10   845     SP      7  1017701        2437   
4  2025    6  21069090       10   365     SP      1   817800        4800   

    Peso  Valor US$  VL_FRETE  VL_SEGURO Nome do País  \
0     13        203       NaN        NaN  El Salvador   
1  83835     111324       NaN        NaN    Singapura   
2    753      22012       NaN        NaN      SuÃ­Ã§a   
3   2437       8458       NaN        NaN      Uruguai   
4   4800      17304       NaN        NaN   IndonÃ©sia   

                                     Nome do Produto Nome do Modal  \
0  Outras pedras preciosas (exceto diamantes) ou ...         AEREA   
1  Coxas com sobrecoxas nÃ£o desossadas de galinh...      MA

In [ ]:
# Exclusão de colunas específicas
# Primeiro, remover as colunas de código originais
colunas_codigos_para_excluir = ["Produto", "País", "Modal", "Aduana"]
df_estado_processado = df_estado_processado.drop(columns=colunas_codigos_para_excluir, errors='ignore')

# Renomear as colunas de nome para os nomes das colunas de código
df_estado_processado = df_estado_processado.rename(columns={
    "Nome do País": "País",
    "Nome do Produto": "Produto",
    "Nome do Modal": "Modal",
    "Nome da Aduana": "Aduana"
})

# Exclusão de outras colunas
colunas_para_excluir_adicionais = ["Unidade", "VL_SEGURO", "VL_FRETE"]
df_estado_processado = df_estado_processado.drop(columns=colunas_para_excluir_adicionais, errors='ignore')

# Relacionar os números com seus respectivos meses.
mapa_mensal = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
}
df_estado_processado['Mês'] = df_estado_processado['Mês'].map(mapa_mensal)

# Visualização do DataFrame editado
display(df_estado_processado.head())

,Ano,Mês,Estado,Quantidade,Peso,Valor US$,País,Produto,Modal,Aduana
0,2025,Agosto,RS,61690,13,203,El Salvador,Outras pedras preciosas (exceto diamantes) ou ...,AEREA,0817700 - AEROPORTO INTERNACIONAL DE VIRACOPOS
1,2025,Dezembro,SC,83835,83835,111324,Singapura,Coxas com sobrecoxas nÃ£o desossadas de galinh...,MARITIMA,0817800 - PORTO DE SANTOS
2,2025,Novembro,SP,792,753,22012,SuÃ­Ã§a,Outras preparaÃ§Ãµes capilares,AEREA,0817600 - AEROPORTO INTERNACIONAL DE SAO PAULO...
3,2025,Novembro,SP,2437,2437,8458,Uruguai,"Falsos tecidos de poliÃ©ster, de peso superior...",RODOVIARIA,1017701 - IRF - CHUÃ
4,2025,Junho,SP,4800,4800,17304,IndonÃ©sia,Outras preparaÃ§Ãµes alimentÃ­cias,MARITIMA,0817800 - PORTO DE SANTOS
